Cleaning up all datasets

Import pandas and initalizing what the final dataframe columns will be.

In [7]:
import pandas as pd

final_columns = ["text", "label", "source_dataset", "dataset_name"]

import re
import pandas as pd

def deep_clean_text(series):
    """
    Applies regex cleaning to a pandas Series of text to remove structural artifacts
    and prevent data leakage before model training.
    """
    # 1. Fill NAs and convert to lowercase string
    s = series.fillna("").astype(str).str.lower()
    
    # 2. Remove URLs
    s = s.str.replace(r'https?://\S+|www\.\S+', '', regex=True)
    
    # 3. Remove HTML tags
    s = s.str.replace(r'<.*?>', '', regex=True)
    
    # 4. Remove Twitter handles (e.g., @username)
    s = s.str.replace(r'@\w+', '', regex=True)
    
    # 5. Remove common news datelines (e.g., "washington (reuters) - ")
    s = s.str.replace(r'^.*?\(reuters\)\s*[-—]\s*', '', regex=True)
    
    # 6. Hard-delete the word "reuters" anywhere else it appears
    s = s.str.replace(r'\breuters\b', '', regex=True)
    
    # 7. Remove punctuation 
    # (Strips out leftover formatting like dashes, brackets, or trailing colons)
    s = s.str.replace(r'[^\w\s]', '', regex=True)
    
    # 8. Strip excess whitespace
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()

    s = s.str.replace(r'\b(reporting by|editing by|our standards:).*$', '', flags=re.IGNORECASE, regex=True)
    
    # 10. Hard-delete residual publisher artifacts
    s = s.str.replace(r'\beikon\b', '', regex=True)
    
    return s

import spacy
import pandas as pd

# Load the model, but disable components we don't need for pure NER. 
# This makes the processing significantly faster.
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])

def mask_entities(text_series):
    """
    Takes a pandas Series of text, finds specific named entities, 
    and replaces them with generic NER tags.
    """
    masked_texts = []
    
    # We only want to mask entities that cause topic bias. 
    # We leave dates, times, and money alone, as those can be stylistic markers.
    target_labels = {"PERSON", "ORG", "GPE", "LOC", "NORP"} 
    # GPE = Geopolitical Entity (Countries, Cities)
    # NORP = Nationalities, Religious, or Political Groups
    
    # nlp.pipe processes the text in batches for speed
    for doc in nlp.pipe(text_series.fillna("").astype(str), batch_size=500):
        new_text = doc.text
        
        # We iterate through entities in reverse order.
        # If we replace from left to right, changing the string length messes up the 
        # character indices for the remaining entities. Working backwards prevents this.
        for ent in reversed(doc.ents):
            if ent.label_ in target_labels:
                start = ent.start_char
                end = ent.end_char
                # Replace the specific word with its generic tag
                new_text = new_text[:start] + f"[{ent.label_}]" + new_text[end:]
                
        masked_texts.append(new_text)
        
    return pd.Series(masked_texts, index=text_series.index)

def clean_text_column(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
    )

def standardize_dataset(df, text_cols, label_col, label_map, source_name, dataset_name):
    df = df.copy()

    # combine one or more text columns into a single text column
    available_text_cols = [col for col in text_cols if col in df.columns]
    if not available_text_cols:
        raise KeyError(f"None of the text columns were found: {text_cols}")
    
    # --- UPDATED SECTION ---
    text_parts = []
    for col in available_text_cols:
        
        cleaned_series = clean_text_column(df[col])
        
        text_parts.append(cleaned_series)
    # -----------------------
    
    df["text"] = text_parts[0]
    for part in text_parts[1:]:
        df["text"] = df["text"] + " " + part

    df["text"] = df["text"].str.strip()

    # standardize label
    label_series = df[label_col]
    if pd.api.types.is_numeric_dtype(label_series):
        normalized_labels = pd.to_numeric(label_series, errors="coerce")
        normalized_map = label_map
    else:
        normalized_labels = label_series.astype(str).str.strip().str.lower()
        normalized_map = {
            str(key).strip().lower(): value
            for key, value in label_map.items()
        }

    df["label"] = normalized_labels.map(normalized_map)
    unmatched_labels = sorted(pd.Series(normalized_labels[df["label"].isna()]).dropna().unique().tolist())
    if unmatched_labels:
        print(f"Unmatched labels for {source_name}: {unmatched_labels[:10]}")

    # add source name
    df["source_dataset"] = source_name
    df["dataset_name"] = dataset_name

    # keep only needed columns
    df = df[final_columns]

    # remove bad rows
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"] != ""]

    # optional: force label to int
    df["label"] = df["label"].astype(int)

    # remove duplicates based on text
    df = df.drop_duplicates(subset=["text"])

    return df

def sample_balanced_by_dataset(df, target_per_label=7500, random_state=42):
    sampled_parts = []

    for label_value in sorted(df["label"].unique()):
        label_df = df[df["label"] == label_value].copy()
        if len(label_df) < target_per_label:
            raise ValueError(
                f"Not enough rows for label {label_value}: "
                f"{len(label_df)} available, {target_per_label} requested"
            )

        counts = label_df["dataset_name"].value_counts().sort_index()
        raw_targets = counts / counts.sum() * target_per_label
        base_targets = raw_targets.astype(int)
        remainders = raw_targets - base_targets

        if (counts >= 1).all() and len(base_targets) <= target_per_label:
            base_targets = base_targets.mask(base_targets == 0, 1)

        overflow = int(base_targets.sum() - target_per_label)
        if overflow > 0:
            for dataset_name in base_targets.sort_values(ascending=False).index:
                reducible = min(overflow, max(0, base_targets[dataset_name] - 1))
                if reducible > 0:
                    base_targets[dataset_name] -= reducible
                    overflow -= reducible
                if overflow == 0:
                    break

        shortfall = int(target_per_label - base_targets.sum())
        if shortfall > 0:
            for dataset_name in remainders.sort_values(ascending=False).index:
                available_extra = counts[dataset_name] - base_targets[dataset_name]
                if available_extra > 0:
                    base_targets[dataset_name] += 1
                    shortfall -= 1
                if shortfall == 0:
                    break

        for dataset_name, n_rows in base_targets.items():
            if n_rows > 0:
                subset = label_df[label_df["dataset_name"] == dataset_name]
                sampled_parts.append(subset.sample(n=int(n_rows), random_state=random_state))

    sampled_df = pd.concat(sampled_parts, ignore_index=True)
    return sampled_df.sample(frac=1, random_state=random_state).reset_index(drop=True)


Load, clean, and combine the requested datasets using the reusable helper functions.

In [8]:
fake_news_test = standardize_dataset(
    pd.read_csv("Data/fake_news_test.csv"),
    text_cols=["text"],
    label_col="label",
    label_map={"Fake": 1, "Real": 0},
    source_name="news",
    dataset_name="fake_news_test"
)

fake_news_train = standardize_dataset(
    pd.read_csv("Data/fake_news_train.csv"),
    text_cols=["text"],
    label_col="label",
    label_map={"Fake": 1, "Real": 0},
    source_name="news",
    dataset_name="fake_news_train"
)

train = standardize_dataset(
    pd.read_csv("Data/train.csv"),
    text_cols=["text"],
    label_col="target",
    label_map={0: 1, 1: 0},
    source_name="twitter",
    dataset_name="train"
)

train1 = standardize_dataset(
    pd.read_csv("Data/train1.csv"),
    text_cols=["text"],
    label_col="label",
    label_map={"fake": 1, "real": 0},
    source_name="news",
    dataset_name="train1"
)

true_news = standardize_dataset(
    pd.read_csv("Data/True.csv").assign(label=0),
    text_cols=["text"],
    label_col="label",
    label_map={0: 0},
    source_name="news",
    dataset_name="True"
)

fake_news_all = standardize_dataset(
    pd.read_csv("Data/Fake.csv").assign(label=1),
    text_cols=["text"],
    label_col="label",
    label_map={1: 1},
    source_name="news",
    dataset_name="Fake"
)

train_tsv_raw = pd.read_csv("Data/train.tsv", sep="\t", header=None)
train_tsv_raw = train_tsv_raw[train_tsv_raw[1].isin(["true", "false"])]
train_tsv = standardize_dataset(
    train_tsv_raw.rename(columns={2: "text", 1: "label"}),
    text_cols=["text"],
    label_col="label",
    label_map={"true": 0, "false": 1},
    source_name="twitter",
    dataset_name="train_tsv"
)

test_tsv_raw = pd.read_csv("Data/test.tsv", sep="\t", header=None)
test_tsv_raw = test_tsv_raw[test_tsv_raw[1].isin(["true", "false"])]
test_tsv = standardize_dataset(
    test_tsv_raw.rename(columns={2: "text", 1: "label"}),
    text_cols=["text"],
    label_col="label",
    label_map={"true": 0, "false": 1},
    source_name="twitter",
    dataset_name="test_tsv"
)

dataset_counts = {
    "fake_news_test": len(fake_news_test),
    "fake_news_train": len(fake_news_train),
    "train": len(train),
    "train1": len(train1),
    "true_news": len(true_news),
    "fake_news_all": len(fake_news_all),
    "train_tsv": len(train_tsv),
    "test_tsv": len(test_tsv),
}
print(pd.Series(dataset_counts).sort_index())

big_dataset = pd.concat(
    [
        fake_news_test,
        fake_news_train,
        train,
        train1,
        true_news,
        fake_news_all,
        train_tsv,
        test_tsv,
    ],
    ignore_index=True
)

big_dataset = big_dataset.drop_duplicates(subset=["text"]).reset_index(drop=True)

less_big_dataset = sample_balanced_by_dataset(big_dataset, target_per_label=7500, random_state=42)

print(big_dataset["source_dataset"].value_counts())
print(big_dataset["dataset_name"].value_counts())
print(big_dataset["label"].value_counts())
print(less_big_dataset["source_dataset"].value_counts())
print(less_big_dataset["dataset_name"].value_counts())
print(less_big_dataset["label"].value_counts())
less_big_dataset.head(15000)


Processing fake_news_test: Masking entities...
Processing fake_news_train: Masking entities...
Processing train: Masking entities...
Processing train1: Masking entities...


KeyboardInterrupt: 

In [6]:
# Assuming you have already loaded or sampled your 15k dataframe
# For example: df = pd.read_csv("Data/smaller_data.csv")

print("Step 1: Masking Named Entities (Running spaCy on 15k rows...)")
# Capitalization is intact here, so spaCy can find the names and places
less_big_dataset["text"] = mask_entities(less_big_dataset["text"])

print("Step 2: Deep Cleaning (Stripping formatting, URLs, and artifacts...)")
# Now that entities are masked, we strip punctuation and lowercase everything
less_big_dataset["text"] = deep_clean_text(less_big_dataset["text"])

print("Step 3: Dropping new duplicates...")
# Cleaning often reveals exact duplicates that were previously hiding behind slight formatting differences
df = less_big_dataset.drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"Done! Final dataset size: {len(df)} rows.")

# Now df is ready for your TF-IDF vectorizer or Hugging Face dataset creation
less_big_dataset.to_csv('Data/smaller_data.csv', index=False)
big_dataset.to_csv('Data/big_data.csv', index=False)